# Phase 1C - Cognitive architectures (12-19)

**Phase 1C - Cognitive architectures (12-19)** - an independent notebook (runnable standalone in Colab or locally).

Covers: Cognitive architectures (12-19) - human-inspired patterns.

Attribution: adapted from *Agent Memory Techniques* by Nir Diamant (https://github.com/NirDiamant/Agent_Memory_Techniques), Apache-2.0. Inline demos are original.

## 0. Setup (self-contained - run this first)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag" # change in scripts/build_phase_notebooks.py to retarget everywhere
try:
 import google.colab # noqa
 IN_COLAB = True
except Exception:
 IN_COLAB = False
if IN_COLAB:
 repo = Path("/content/kdd26-memdiag")
 if not repo.exists():
 subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
 SOURCE = repo / "experiment" / "github_submission" / "source"
 subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
 SOURCE = None
 for cand in [Path.cwd(), *Path.cwd().parents]:
 for sub in ("source", "experiment"):
 if (cand / sub / "run.py").exists():
 SOURCE = cand / sub
 break
 if SOURCE:
 break
 if SOURCE is None:
 raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

from embedder import TfidfHashEmbedder
from llm_client import OfflineLLMClient
from memory_core import MemoryRecord
from providers import build_provider
emb = TfidfHashEmbedder()
try:
 from llm_client import LLMConfig, make_client
 llm = make_client(LLMConfig(backend='openai-compatible' if os.environ.get('OPENAI_API_KEY') else 'offline',
 base_url=os.environ.get('OPENAI_BASE_URL','https://api.openai.com/v1'),
 model='gpt-4o'))
except Exception:
 llm = OfflineLLMClient()
print('LLM backend:', llm.backend, '| API key:', bool(os.environ.get('OPENAI_API_KEY')))
TURNS = [
 ("Alice", "Hi, I am Alice. I work as a data scientist at a health-tech startup in Berlin."),
 ("Bob", "I am Bob, an ML engineer in Athens. I prefer PyTorch."),
 ("Alice", "We deploy on Kubernetes and track runs with Weights and Biases."),
 ("Bob", "Our training run failed last night with CUDA OOM at batch 256."),
 ("Alice", "We hit that before. Reducing batch to 64 and enabling gradient checkpointing fixed it."),
 ("Bob", "Our best val_loss was 0.423 with lr=3e-4 and weight_decay=0.01."),
 ("Alice", "I live in Prenzlauer Berg. My favorite coffee shop is on Kollwitzplatz."),
 ("Bob", "Let us sync next Tuesday at 10am CET."),
]
records = [MemoryRecord(f"t{i}", f"{w}: {t}", {"session_id": "s1" if i < 4 else "s2"}) for i, (w, t) in enumerate(TURNS)]
print("setup OK | turns =", len(TURNS))


## Cognitive architectures (12-19) - human-inspired patterns

**12 - Working Memory** · Pin/evict context slots by priority

In [ ]:
# Technique 12 - Working Memory: Pin/evict context slots by priority
pinned=[t for w,t in TURNS if 'OOM' in t][:2]
print('12 working memory pins:', [p[:30] for p in pinned])

**13 - Hierarchical Layers** · Hot/warm/cold tiers; promote/demote

In [ ]:
# Technique 13 - Hierarchical Layers: Hot/warm/cold tiers; promote/demote
p=build_provider('hybrid',emb,llm); p.ingest(records); print('13 hierarchical tiers size:', p.size())

**14 - Consolidation** · Merge/dedup/strengthen memories

In [ ]:
# Technique 14 - Consolidation: Merge/dedup/strengthen memories
seen={}
for w,t in TURNS: seen[t[:12]]=seen.get(t[:12],0)+1
print('14 consolidation: merged to', len(seen), 'unique prefixes')

**15 - Compaction** · Compress via summary/entity/distill

In [ ]:
# Technique 15 - Compaction: Compress via summary/entity/distill
compact=' '.join(t.split()[0] for _,t in TURNS)
print('15 compaction:', compact[:60])

**16 - Self-Reflection** · Agent writes notes on its own actions

In [ ]:
# Technique 16 - Self-Reflection: Agent writes notes on its own actions
notes=['batch 256 -> OOM (avoid)','lr 3e-4 worked']
print('16 self-reflection notes:', notes)

**17 - Routing** · Pick the memory store by content/intent

In [ ]:
# Technique 17 - Routing: Pick the memory store by content/intent
def route(q): return 'temporal' if 'when' in q.lower() else 'semantic'
print('17 routing -> ', route('When do they sync?'))

**18 - Temporal** · Timestamp + recency-weighted retrieval

In [ ]:
# Technique 18 - Temporal: Timestamp + recency-weighted retrieval
import time; now=time.time()
tagged=[(w, round(now-i*1000,0)) for i,(w,t) in enumerate(TURNS)]
print('18 temporal newest:', tagged[-1][0], 'ts', tagged[-1][1])

**19 - Forgetting & Decay** · Prune by decay/access/relevance

In [ ]:
# Technique 19 - Forgetting & Decay: Prune by decay/access/relevance
import time; now=time.time()
mem=[(w, now-i*5000) for i,(w,t) in enumerate(TURNS)]
keep=[m for m in mem if now-m[1]<6000]
print('19 forgetting:', len(mem),'->',len(keep),'after 6000s half-life')